In [17]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import itertools
from scipy import stats
import matplotlib.pyplot as plt

In [18]:
input_file = 'india_monthly_exports_with_indicators.xlsx'

df = pd.read_excel(input_file, sheet_name='Complete Data')
df.columns = df.columns.str.strip()
df['Date'] = pd.to_datetime(df['Date'])

monthly = (df.groupby('Date')
             .agg(Export_Value   = ('Export_Value',   'sum'),
                  GDP_Growth      = ('gdp_growth',      'first'),
                  Inflation_Rate = ('Inflation_rate',  'first'),
                  REER           = ('REER',            'first'))
             .reset_index()
             .sort_values('Date'))
monthly.set_index('Date', inplace=True)
monthly.index = pd.DatetimeIndex(monthly.index).to_period('M')

print("RAW DATA SUMMARY")
print(f" Observations : {len(monthly)}")
print(f" Date range   : {monthly.index[0]} to {monthly.index[-1]}")
print(f" NaN counts   : {dict(monthly.isnull().sum())}")

RAW DATA SUMMARY
 Observations : 96
 Date range   : 2018-01 to 2025-12
 NaN counts   : {'Export_Value': np.int64(0), 'GDP_Growth': np.int64(0), 'Inflation_Rate': np.int64(0), 'REER': np.int64(0)}


### Interpolation of GDP

In [19]:
# INTERPOLATE QUARTERLY GDP growth VALUES TO MONTHLY
monthly['GDP_Growth'] = (
    monthly['GDP_Growth']
    .where(monthly['GDP_Growth'] != monthly['GDP_Growth'].shift(1))
    .interpolate(method='linear')
)

print("INTERPOLATED DATA SUMMARY")
print(f"nan : {monthly['GDP_Growth'].isnull().sum()}")
print(f"\n  First 6 months of GDP Growth after interpolation:")
for idx, val in monthly['GDP_Growth'].head(6).items():
    print(f"{idx}:{val:.4f}")


INTERPOLATED DATA SUMMARY
nan : 0

  First 6 months of GDP Growth after interpolation:
2018-01:7.7000
2018-02:7.7667
2018-03:7.8333
2018-04:7.9000
2018-05:7.6333
2018-06:7.3667


### Data Stationary check

In [20]:
#ADF TEST FUNCTION
def run_adf(series, label, alpha=0.05):
    clean = series.dropna()
    stat, pval, lags, nobs, crit, _ = adfuller(clean, autolag='AIC')
    stationary = pval < alpha
    status  = "STATIONARY" if stationary else "NON-STATIONARY"
    bracket = f"(p={pval:.4f} < alpha={alpha})" if stationary else f"(p={pval:.4f} >= alpha={alpha})"

    print(f"\n Variable: {label}")
    print(f" Series length: {len(clean)}")
    print(f"ADF Statistic: {stat:.6f}")
    print(f"p-value: {pval:.6f}")
    print(f"Lags used: {lags}")
    print(f"Critical vals: 1% {crit['1%']:.4f}|5% {crit['5%']:.4f}|10% {crit['10%']:.4f}")
    print(f"Decision: {status} {bracket}")

    return stationary, pval, lags, nobs


stat_exp,_, _, _ = run_adf(monthly['Export_Value'],'Export_Value(Level)')
stat_gdp,_, _, _ = run_adf(monthly['GDP_Growth'],'GDP_Growth(Level)')
stat_inf,_, _, _ = run_adf(monthly['Inflation_Rate'],'Inflation_Rate(Level)')
stat_reer,_, _, _ = run_adf(monthly['REER'],'REER (Level)')


 Variable: Export_Value(Level)
 Series length: 96
ADF Statistic: -2.015150
p-value: 0.279970
Lags used: 1
Critical vals: 1% -3.5019|5% -2.8928|10% -2.5835
Decision: NON-STATIONARY (p=0.2800 >= alpha=0.05)

 Variable: GDP_Growth(Level)
 Series length: 96
ADF Statistic: -3.902032
p-value: 0.002020
Lags used: 4
Critical vals: 1% -3.5043|5% -2.8939|10% -2.5840
Decision: STATIONARY (p=0.0020 < alpha=0.05)

 Variable: Inflation_Rate(Level)
 Series length: 96
ADF Statistic: -1.577302
p-value: 0.494985
Lags used: 12
Critical vals: 1% -3.5117|5% -2.8970|10% -2.5857
Decision: NON-STATIONARY (p=0.4950 >= alpha=0.05)

 Variable: REER (Level)
 Series length: 96
ADF Statistic: -2.591507
p-value: 0.094765
Lags used: 1
Critical vals: 1% -3.5019|5% -2.8928|10% -2.5835
Decision: NON-STATIONARY (p=0.0948 >= alpha=0.05)


In [21]:
# adf on differenced export values
d_order = 0
stat_exp_d1 = stat_exp_d2 = False

if stat_exp:
    d_order = 0
    print("\nExport_Value is stationary at level I(0), d = 0")
else:
    print("\n  Export_Value non-stationary at level,testing 1st difference")
    stat_exp_d1, _, _, _ = run_adf(
        monthly['Export_Value'].diff(), 'Export_Value (1st Difference)'
    )
    if stat_exp_d1:
        d_order = 1
    else:
        print("\n Still non-stationary,testing 2nd difference")
        stat_exp_d2, _, _, _ = run_adf(
            monthly['Export_Value'].diff().diff(), 'Export_Value (2nd Difference)'
        )
        if stat_exp_d2:
            d_order = 2
        else:
            print(" 2nd difference still non-stationary")

print(f"\nExport_Value is I({d_order})|d_order = {d_order}")

# adf on differenced exogeneous variables
stat_inf_d1,  _, _, _ = run_adf(
    monthly['Inflation_Rate'].diff(), 'Inflation_Rate (1st Difference)'
)
stat_reer_d1, _, _, _ = run_adf(
    monthly['REER'].diff(),'REER(1st Difference)'
)


  Export_Value non-stationary at level,testing 1st difference

 Variable: Export_Value (1st Difference)
 Series length: 95
ADF Statistic: -9.628958
p-value: 0.000000
Lags used: 1
Critical vals: 1% -3.5027|5% -2.8932|10% -2.5836
Decision: STATIONARY (p=0.0000 < alpha=0.05)

Export_Value is I(1)|d_order = 1

 Variable: Inflation_Rate (1st Difference)
 Series length: 95
ADF Statistic: -3.297942
p-value: 0.014971
Lags used: 11
Critical vals: 1% -3.5117|5% -2.8970|10% -2.5857
Decision: STATIONARY (p=0.0150 < alpha=0.05)

 Variable: REER(1st Difference)
 Series length: 95
ADF Statistic: -8.822451
p-value: 0.000000
Lags used: 0
Critical vals: 1% -3.5019|5% -2.8928|10% -2.5835
Decision: STATIONARY (p=0.0000 < alpha=0.05)


In [22]:
export_d1 = monthly['Export_Value'].diff()
gdp_lvl   = monthly['GDP_Growth']
inf_d1    = monthly['Inflation_Rate'].diff()
reer_d1   = monthly['REER'].diff()

data_ready = pd.concat(
    [export_d1, gdp_lvl, inf_d1, reer_d1], axis=1
)
data_ready.columns = ['Export_d1', 'GDP_level', 'Inflation_d1', 'REER_d1']
data_ready = data_ready.dropna()

# Grid serach for best model

In [23]:
# Split Y and exog
Y    = data_ready['Export_d1']
exog = data_ready[['GDP_level', 'Inflation_d1', 'REER_d1']]

In [24]:
N_TEST  = 12
Y_train = Y.iloc[:-N_TEST]
Y_test  = Y.iloc[-N_TEST:]
X_train = exog.iloc[:-N_TEST]
X_test  = exog.iloc[-N_TEST:]

In [28]:
# Grid search  on traing data for BEST ARIMAX(p,1,q)
results = []
for p, q in itertools.product(range(0, 4), range(0, 4)):
    try:
        fit = SARIMAX(Y_train, exog=X_train, order=(p, 0, q),
                      trend='n', enforce_stationarity=True,
                      enforce_invertibility=True).fit(disp=False)
        results.append({'p':p,'q':q,'AIC':round(fit.aic,2),
                        'BIC':round(fit.bic,2),'HQIC':round(fit.hqic,2),
                        'LogLik':round(fit.llf,2),'Converged':True})
        print(f"  ARIMAX({p},1,{q})  AIC={fit.aic:>10.2f}  BIC={fit.bic:>10.2f}")
    except Exception as e:
        results.append({'p':p,'q':q,'AIC':np.nan,'BIC':np.nan,
                        'HQIC':np.nan,'LogLik':np.nan,'Converged':False})
        print(f"  ARIMAX({p},1,{q})  FAILED: {str(e)[:50]}")

grid_df = (pd.DataFrame(results).dropna(subset=['AIC'])
             .sort_values('AIC').reset_index(drop=True))
best_p  = int(grid_df.iloc[0].p)
best_q  = int(grid_df.iloc[0].q)

print(f"\n  Top 5 models:")
print(grid_df.head(5).to_string(index=False))
print(f"\n  Selected: ARIMAX({best_p},1, {best_q})  "
      f"AIC={grid_df.iloc[0].AIC}  BIC={grid_df.iloc[0].BIC}")

  ARIMAX(0,1,0)  AIC=   1600.76  BIC=   1610.44
  ARIMAX(0,1,1)  AIC=   1582.02  BIC=   1594.12
  ARIMAX(0,1,2)  AIC=   1583.50  BIC=   1598.01
  ARIMAX(0,1,3)  AIC=   1585.43  BIC=   1602.36
  ARIMAX(1,1,0)  AIC=   1585.04  BIC=   1597.14
  ARIMAX(1,1,1)  AIC=   1583.35  BIC=   1597.86
  ARIMAX(1,1,2)  AIC=   1585.43  BIC=   1602.36
  ARIMAX(1,1,3)  AIC=   1586.47  BIC=   1605.82
  ARIMAX(2,1,0)  AIC=   1581.62  BIC=   1596.13
  ARIMAX(2,1,1)  AIC=   1583.10  BIC=   1600.04
  ARIMAX(2,1,2)  AIC=   1584.54  BIC=   1603.89
  ARIMAX(2,1,3)  AIC=   1583.17  BIC=   1604.94
  ARIMAX(3,1,0)  AIC=   1582.90  BIC=   1599.83
  ARIMAX(3,1,1)  AIC=   1584.92  BIC=   1604.27
  ARIMAX(3,1,2)  AIC=   1583.46  BIC=   1605.23
  ARIMAX(3,1,3)  AIC=   1586.12  BIC=   1610.30

  Top 5 models:
 p  q     AIC     BIC    HQIC  LogLik  Converged
 2  0 1581.62 1596.13 1587.45 -784.81       True
 0  1 1582.02 1594.12 1586.88 -786.01       True
 3  0 1582.90 1599.83 1589.70 -784.45       True
 2  1 1583.10 1600.

# ARIMAX model fit

In [29]:
final_model = SARIMAX(
    Y, exog=exog,
    order=(best_p, 0, best_q),
    trend='n',
    enforce_stationarity=True,
    enforce_invertibility=True
).fit(disp=False)
print(final_model.summary())

                               SARIMAX Results                                
Dep. Variable:              Export_d1   No. Observations:                   95
Model:               SARIMAX(2, 0, 0)   Log Likelihood                -895.655
Date:                Sun, 19 Apr 2026   AIC                           1803.310
Time:                        22:06:05   BIC                           1818.634
Sample:                    02-28-2018   HQIC                          1809.502
                         - 12-31-2025                                         
Covariance Type:                  opg                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
GDP_level       49.4704     23.227      2.130      0.033       3.947      94.994
Inflation_d1  1083.3164    477.667      2.268      0.023     147.106    2019.526
REER_d1       -409.1016    251.908     -1.62

# In-sample Validation

In [40]:
fitted_diff = final_model.fittedvalues
prev_lvl    = monthly['Export_Value'].shift(1)
common_idx  = fitted_diff.index.intersection(prev_lvl.dropna().index)

y_true_is = monthly['Export_Value'].loc[common_idx].values
y_pred_is = (prev_lvl.loc[common_idx] + fitted_diff.loc[common_idx]).values

rmse_is = np.sqrt(mean_squared_error(y_true_is, y_pred_is))
mae_is  = mean_absolute_error(y_true_is, y_pred_is)
mape_is = np.mean(np.abs((y_true_is - y_pred_is) / y_true_is)) * 100
r2_is   = 1 - np.sum((y_true_is-y_pred_is)**2) / np.sum((y_true_is-y_true_is.mean())**2)

print(f"\nIn-Sample  (n={len(y_true_is)}, one-step-ahead on level):")
print(f"  RMSE : {rmse_is:>10.2f}  USD M")
print(f"  MAE  : {mae_is:>10.2f}  USD M")
print(f"  MAPE : {mape_is:>10.2f}  %")
print(f"  R²   : {r2_is:>10.4f}")


In-Sample  (n=95, one-step-ahead on level):
  RMSE :    3003.17  USD M
  MAE  :    2381.92  USD M
  MAPE :       8.46  %
  R²   :     0.7646


In [41]:
print(f"\n  {'Month':<12} {'Actual (USD M)':>14} {'Predicted':>12} {'Error':>10}")

for i, m in enumerate(monthly.index[-N_TEST:]):
    err = actual_lvl[i] - pred_lvl_oos[i]
    print(f"  {str(m):<12} {actual_lvl[i]:>14.2f} {pred_lvl_oos[i]:>12.2f} {err:>10.2f}")



  Month        Actual (USD M)    Predicted      Error
  2025-01            36338.46     36975.30    -636.84
  2025-02            36910.95     36521.59     389.36
  2025-03            42047.58     38053.89    3993.69
  2025-04            38283.21     42013.30   -3730.09
  2025-05            38303.31     38483.35    -180.04
  2025-06            34969.76     38659.38   -3689.62
  2025-07            37017.55     34810.28    2207.27
  2025-08            34770.55     38412.76   -3642.21
  2025-09            36132.77     35169.51     963.26
  2025-10            34105.71     35186.69   -1080.98
  2025-11            37908.48     34921.23    2987.25
  2025-12            38468.38     39967.45   -1499.07


# Out of sample  check

In [42]:
oos_fit    = SARIMAX(Y_train, exog=X_train, order=(best_p, 0, best_q),
                     trend='n', enforce_stationarity=True,
                     enforce_invertibility=True).fit(disp=False)
oos_fc     = oos_fit.get_forecast(steps=N_TEST, exog=X_test)
oos_pred_d = oos_fc.predicted_mean.values

actual_prev  = monthly['Export_Value'].iloc[-(N_TEST+1):-1].values  # actual[t-1]
actual_lvl   = monthly['Export_Value'].iloc[-N_TEST:].values         # actual[t]
pred_lvl_oos = actual_prev + oos_pred_d                              # one-step inversion

rmse_oos = np.sqrt(mean_squared_error(actual_lvl, pred_lvl_oos))
mae_oos  = mean_absolute_error(actual_lvl, pred_lvl_oos)
mape_oos = np.mean(np.abs((actual_lvl - pred_lvl_oos) / actual_lvl)) * 100

print(f"\nOut-of-Sample  (last n={N_TEST} months, one-step-ahead on level):")
print(f"  RMSE : {rmse_oos:>10.2f}  USD M")
print(f"  MAE  : {mae_oos:>10.2f}  USD M")
print(f"  MAPE : {mape_oos:>10.2f}  %")


Out-of-Sample  (last n=12 months, one-step-ahead on level):
  RMSE :    2507.84  USD M
  MAE  :    2083.31  USD M
  MAPE :       5.59  %


In [43]:
last_gdp  = monthly['GDP_Growth'].iloc[-1]
fut_exog  = pd.DataFrame({
    'GDP_level':    [last_gdp] * 12,
    'Inflation_d1': [0.0] * 12,
    'REER_d1':      [0.0] * 12
})
print(f"\n  Exog assumptions:")
print(f"    GDP_level    = {last_gdp:.4f}  (last observed, constant)")
print(f"    Inflation_d1 = 0.0000  (Inflation level flat, no monthly change)")
print(f"    REER_d1      = 0.0000  (REER level flat, no monthly change)")

fut_fc  = final_model.get_forecast(steps=12, exog=fut_exog)
fut_d   = fut_fc.predicted_mean
fut_ci  = fut_fc.conf_int(alpha=0.05)

last_export  = monthly['Export_Value'].iloc[-1]
fut_level    = last_export + fut_d.cumsum()
fut_lo       = last_export + fut_ci.iloc[:, 0].cumsum()
fut_hi       = last_export + fut_ci.iloc[:, 1].cumsum()
fut_idx      = pd.period_range(start=monthly.index[-1]+1, periods=12, freq='M')

print(f"\n  {'Month':<12} {'Forecast (USD M)':>16} {'Lower 95%':>14} {'Upper 95%':>14}")
print(f"  {'-'*58}")
for i, p in enumerate(fut_idx):
    print(f"  {str(p):<12} {fut_level.iloc[i]:>16.2f}"
          f" {fut_lo.iloc[i]:>14.2f} {fut_hi.iloc[i]:>14.2f}")

print(f"\n  CI widens with horizon -- uncertainty compounds over multi-step forecast.")


  Exog assumptions:
    GDP_level    = 7.6000  (last observed, constant)
    Inflation_d1 = 0.0000  (Inflation level flat, no monthly change)
    REER_d1      = 0.0000  (REER level flat, no monthly change)

  Month        Forecast (USD M)      Lower 95%      Upper 95%
  ----------------------------------------------------------
  2026-01              38828.41       32895.61       44761.21
  2026-02              39536.91       26840.83       52232.99
  2026-03              39734.81       20267.88       59201.73
  2026-04              40126.57       13860.24       66392.90
  2026-05              40537.65        7459.00       73616.29
  2026-06              40890.53         999.13       80781.93
  2026-07              41270.52       -5433.90       87974.94
  2026-08              41649.97      -11867.67       95167.62
  2026-09              42023.06      -18307.82      102353.94
  2026-10              42399.76      -24744.36      109543.88
  2026-11              42776.05      -31181.32   